# Lab Activity 6: Multi-Class Classification — SVM vs Neural Network
**Course:** CSE473: Computational Intelligence — Mechatronics Engineering and Automation Program
**Prepares you for:** Lab Assignment 06 — Double Moon Dataset, Four-Category Classification

## 🎯 Learning objectives
By the end of this lab you will be able to:
- Generate a **four-category** double moon dataset and one-hot encode labels
- Shuffle-split data 60/20/20 and keep every class represented
- Train a multi-class **SVM** (`sklearn.svm.SVC`, RBF kernel) and interpret its decision regions
- Train a **softmax neural network** in pure numpy (cross-entropy, standardization, momentum)
- Compare the two models on accuracy and per-class behavior — the assignment's workflow

⏱ **Estimated time: ~55 minutes**

## How this lab works
- The lab is split into **Parts**; each Part teaches one topic.
- Each Part starts with a short explanation plus a runnable **✏️ Worked example** — run it, tweak it, break it.
- Then you solve **🎯 Problems**. Read the task, write your code in the starter cell, and try it before opening any hints.
- Stuck? Open the **💡 Hint** blocks below each problem — Hint 1 is a nudge, Hint 2 names the approach. They get more specific as you go.
- Verify yourself with the **🧪 Self-check** cells — they run deterministic checks and fail with guiding messages until your solution is right.
- Truly stuck? The **✅ Reveal solution** block at the bottom of each hint section shows full working code.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
import sklearn

print("NumPy version:", np.__version__)
print("scikit-learn version:", sklearn.__version__)
print("Setup OK — NumPy, matplotlib and scikit-learn are ready.")

## Part 1: Four moons & one-hot labels (≈12 min)

The four-category dataset is **two moon pairs side by side**:

- class 0: upward crescent at $(0, 0)$
- class 1: downward crescent at $(\text{radius}, d)$
- class 2: upward crescent at $(\text{shift}, 0)$
- class 3: downward crescent at $(\text{shift} + \text{radius}, d)$

Each crescent is the same half-annulus as in Lab Activity 5 (radii in $[\text{radius} - \text{width}/2, \text{radius} + \text{width}/2]$, angles in $(0, \pi)$, negated for the downward moons). Labels live in $\{0, 1, 2, 3\}$, and a neural network wants them **one-hot**: row $i$ of $Y$ is all zeros except a 1 in column $y_i$.

In [ ]:
# --- Worked example: sketch the four-moon layout ---
rng = np.random.default_rng(7)

def sketch_moon(n, cx, cy, down):
    r = 10 - 2 + 4 * rng.random(n)      # radius=10, width=4 -> band [8, 12]
    th = np.pi * rng.random(n)
    sgn = -1.0 if down else 1.0
    return np.column_stack([cx + sgn * r * np.cos(th), cy + sgn * r * np.sin(th)])

sketch = [(sketch_moon(120, 0, 0, False), "tomato"),
          (sketch_moon(120, 10, 7, True), "steelblue"),
          (sketch_moon(120, 25, 0, False), "orange"),
          (sketch_moon(120, 35, 7, True), "seagreen")]

fig, ax = plt.subplots(figsize=(9, 4))
for pts, c in sketch:
    ax.scatter(pts[:, 0], pts[:, 1], s=6, c=c)
ax.set_title("four interleaved crescents: two moon pairs side by side")
ax.set_aspect("equal"); ax.grid(alpha=0.3)
plt.show()

## 🎯 Problem 6.1 — generate_four_moons

**Given:** points per class and the moon geometry parameters.

**Required:** write `generate_four_moons(n_per_class=150, radius=10, width=4, d=7, shift=25, seed=7)` returning `(X, y)` with the layout from the concept text — moon $k$ generated with seed `seed + k`.

**Expected output:** `X` of shape $(4\,\text{n\_per\_class}, 2)$; each class label appears exactly `n_per_class` times; classes 0/2 live in the radial band of their upward crescent ($y \geq$ center), classes 1/3 in their downward crescent ($y \leq d$). This mirrors `generate_double_moon_four_categories` in **Lab Assignment 06**.

In [ ]:
def generate_four_moons(n_per_class=150, radius=10, width=4, d=7, shift=25, seed=7):
    """Generate the four-category double-moon dataset (two moon pairs).

    Args:
        n_per_class: points in each of the four classes
        radius: outer radius of every crescent
        width: radial thickness of every crescent
        d: vertical offset of the downward moons' centers
        shift: horizontal offset of the right moon pair
        seed: reproducibility seed (class k uses seed + k)

    Returns:
        (X, y): X of shape (4*n_per_class, 2); y in {0, 1, 2, 3}
    """
    # TODO: Your code here
    pass

# Demo call
demo_4m = generate_four_moons(100)
if demo_4m is not None:
    X6d, y6d = demo_4m
    print("X shape:", X6d.shape, "| per-class counts:", np.bincount(y6d))
else:
    print("Implement generate_four_moons to power this demo.")

<details>
<summary>💡 Hint 1 — four crescents, four seeds</summary>

Loop over the four (center, downward) pairs from the concept text. Inside the loop, draw radii in [radius - width/2, radius + width/2] and angles in (0, pi) from np.random.default_rng(seed + k), negate both coordinates when downward, and stack the results.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
specs = [((0.0, 0.0), False), ((radius, d), True),
         ((shift, 0.0), False), ((shift + radius, d), True)]
for k, ((cx, cy), down) in enumerate(specs):
    rng = np.random.default_rng(seed + k)
    r = radius - width / 2 + width * rng.random(n_per_class)
    theta = rng.random(n_per_class) * np.pi
    sgn = -1.0 if down else 1.0
    Xs.append(np.column_stack([cx + sgn * r * cos(theta), cy + sgn * r * sin(theta)]))
    ys.append(np.full(n_per_class, k, dtype=int))
return np.vstack(Xs), np.concatenate(ys)
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def generate_four_moons(n_per_class=150, radius=10, width=4, d=7, shift=25, seed=7):
    """Generate the four-category double-moon dataset (two moon pairs)."""
    specs = [((0.0, 0.0), False), ((float(radius), float(d)), True),
             ((float(shift), 0.0), False), ((float(shift + radius), float(d)), True)]
    Xs, ys = [], []
    for k, ((cx, cy), down) in enumerate(specs):
        rng = np.random.default_rng(seed + k)
        r = radius - width / 2 + width * rng.random(n_per_class)
        theta = rng.random(n_per_class) * np.pi
        sgn = -1.0 if down else 1.0
        Xs.append(np.column_stack([cx + sgn * r * np.cos(theta),
                                   cy + sgn * r * np.sin(theta)]))
        ys.append(np.full(n_per_class, k, dtype=int))
    return np.vstack(Xs), np.concatenate(ys)
```
</details>

In [ ]:
# 🧪 Self-check for Problem 6.1
d61 = generate_four_moons(120, seed=7)
assert d61 is not None, "❌ generate_four_moons returned None — replace the 'pass'. See Hint 2 in Part 1."
X61 = np.asarray(d61[0], dtype=float); y61 = np.asarray(d61[1])
assert X61.shape == (480, 2), f"❌ X must have shape (4*n_per_class, 2) = (480, 2), got {X61.shape}."
assert np.array_equal(np.bincount(y61, minlength=4), [120, 120, 120, 120]), "❌ Each class must appear exactly n_per_class times — stack four moons of equal size."
assert set(np.unique(y61)).issubset({0, 1, 2, 3}), "❌ Labels must lie in {0, 1, 2, 3}."
centers61 = [(0.0, 0.0), (10.0, 7.0), (25.0, 0.0), (35.0, 7.0)]
for cls in range(4):
    P61 = X61[y61 == cls]
    rr61 = np.hypot(P61[:, 0] - centers61[cls][0], P61[:, 1] - centers61[cls][1])
    assert rr61.min() >= 8.0 - 1e-9 and rr61.max() <= 12.0 + 1e-9, f"❌ Class {cls} must lie in the radial band [radius - width/2, radius + width/2] = [8, 12] around {centers61[cls]}."
assert X61[y61 == 0][:, 1].min() >= -1e-9 and X61[y61 == 2][:, 1].min() >= -1e-9, "❌ Classes 0 and 2 are UPWARD crescents — y must stay at or above their centers."
assert X61[y61 == 1][:, 1].max() <= 7.0 + 1e-9 and X61[y61 == 3][:, 1].max() <= 7.0 + 1e-9, "❌ Classes 1 and 3 are DOWNWARD crescents — y must stay at or below d."
d61b = generate_four_moons(120, seed=7)
assert np.allclose(X61, np.asarray(d61b[0], dtype=float)), "❌ Same seed must reproduce the dataset."
print("✅ Problem 6.1 passed — four moons generated.")

## 🎯 Problem 6.2 — one_hot

**Given:** an integer label array `y` and a class count.

**Required:** write `one_hot(y, num_classes=4)` returning a float array of shape `(len(y), num_classes)` where row $i$ is 1 in column `y[i]` and 0 elsewhere. Two lines with `np.zeros` and fancy indexing do it.

**Expected output:** rows sum to 1, `Y[i, y[i]] == 1`, and `Y.argmax(axis=1)` recovers `y` — the self-check verifies all three.

In [ ]:
def one_hot(y, num_classes=4):
    """One-hot encode integer labels.

    Args:
        y: integer labels, values in [0, num_classes)
        num_classes: number of classes

    Returns:
        float array of shape (len(y), num_classes)
    """
    # TODO: Your code here
    pass

# Demo call
demo_oh = one_hot(np.array([0, 2, 3, 2]))
if demo_oh is not None:
    print(demo_oh)
else:
    print("Implement one_hot to power this demo.")

<details>
<summary>💡 Hint 1 — zeros, then fancy indexing</summary>

Allocate a zeros matrix of shape (len(y), num_classes), then place a 1.0 in each row at the column given by the label. `Y[np.arange(len(y)), y] = 1.0` does all rows at once.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
Y = np.zeros((len(y), num_classes))
Y[np.arange(len(y)), y] = 1.0
return Y
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def one_hot(y, num_classes=4):
    """One-hot encode integer labels."""
    Y = np.zeros((len(y), num_classes))
    Y[np.arange(len(y)), y] = 1.0
    return Y
```
</details>

In [ ]:
# 🧪 Self-check for Problem 6.2
y62 = np.array([0, 3, 1, 2, 2, 0, 3])
Y62 = one_hot(y62)
assert Y62 is not None, "❌ one_hot returned None — replace the 'pass'. See Hint 2 in Part 1."
Y62 = np.asarray(Y62, dtype=float)
assert Y62.shape == (7, 4), f"❌ Expected shape (7, 4), got {Y62.shape}."
assert np.allclose(Y62.sum(axis=1), 1.0), "❌ Every one-hot row must sum to exactly 1."
assert np.allclose(Y62[np.arange(7), y62], 1.0), "❌ Entry [i, y[i]] must be 1 — use Y[np.arange(len(y)), y] = 1.0."
assert np.array_equal(Y62.argmax(axis=1), y62), "❌ argmax over rows must recover the original labels."
assert (Y62 == 0).sum() == 7 * 3, "❌ All entries except the single 1 per row must be 0."
assert np.issubdtype(np.asarray(one_hot(np.array([1]))).dtype, np.floating), "❌ Return a float array."
print("✅ Problem 6.2 passed — labels one-hot encoded.")

## Part 2: Splitting & the multi-class SVM (≈12 min)

The split is the same 60/20/20 shuffle as in Lab Activity 5 — the worked example below defines `train_validation_test_split` again, ready to reuse in every later part. (Train on 60%, tune on 20%, grade on 20% — with balanced source data every split still sees all four classes.)

The **SVM** looks for the boundary with the widest **margin** between classes; the **RBF kernel** $k(x, x') = e^{-\gamma\|x - x'\|^2}$ lets it carve curved, localized regions. `sklearn.svm.SVC` handles multi-class automatically (one-vs-one); `C` trades margin width against training errors, `gamma` sets the kernel's reach. Unlike gradient descent, an SVM is not iterative — there is no loss curve, only a final error value.

In [ ]:
# --- Worked example: the split helper (reuse this in later parts) + a first SVM ---
def train_validation_test_split(X, y, train_frac=0.6, val_frac=0.2, seed=7):
    """Shuffle and split (X, y) into train / validation / test subsets."""
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(X))
    n_train = int(train_frac * len(X))
    n_val = int(val_frac * len(X))
    tr, va, te = idx[:n_train], idx[n_train:n_train + n_val], idx[n_train + n_val:]
    return (X[tr], y[tr]), (X[va], y[va]), (X[te], y[te])

# tiny two-blob demo of the SVC mechanics
rng = np.random.default_rng(0)
Xa = rng.normal((2, 2), 0.5, size=(40, 2))
Xb = rng.normal((6, 6), 0.5, size=(40, 2))
Xdemo = np.vstack([Xa, Xb]); ydemo = np.array([0] * 40 + [1] * 40)
svm_demo = SVC(kernel="rbf", C=1.0).fit(Xdemo, ydemo)
print("SVM training accuracy on the blob demo:", svm_demo.score(Xdemo, ydemo))
print("predicted labels:", svm_demo.predict(Xdemo[:6]))

## 🎯 Problem 6.3 — train_mcsvm

**Given:** train/val/test arrays and hyperparameters `C`, `gamma`.

**Required:** write `train_mcsvm(X_train, y_train, X_val, y_val, X_test, y_test, C=10.0, gamma="scale")` that fits `SVC(kernel="rbf", C=C, gamma=gamma)` on the training set and returns a dict with keys `'model'`, `'train_acc'`, `'val_acc'`, `'test_acc'`, `'train_error'`, `'val_error'` — the errors are $1 - $ accuracy (a single final value each, since an SVM has no iterations).

**Expected output:** on 100-per-class moons, test accuracy lands around 0.93–0.97. This is **Lab Assignment 06** Task 2 in one function.

In [ ]:
def train_mcsvm(X_train, y_train, X_val, y_val, X_test, y_test, C=10.0, gamma="scale"):
    """Train a multi-class SVM (RBF) and report accuracies / errors.

    Args:
        X_train, y_train: training data
        X_val, y_val: validation data
        X_test, y_test: test data
        C: SVM regularization parameter
        gamma: RBF kernel coefficient ("scale" is a fine default)

    Returns:
        dict with keys 'model', 'train_acc', 'val_acc', 'test_acc',
        'train_error', 'val_error'
    """
    # TODO: Your code here
    pass

# Demo call (uses your Problem 6.1 data once implemented)
demo_63 = generate_four_moons(80, seed=3)
if demo_63 is not None:
    X63, y63 = demo_63
    s63 = train_validation_test_split(X63, y63, seed=3)
    if s63 is not None:
        (a63, b63), (c63, d63), (e63, f63) = s63
        demo_svm = train_mcsvm(a63, b63, c63, d63, e63, f63)
        if demo_svm is not None:
            print("SVM test accuracy:", round(demo_svm["test_acc"], 3))
        else:
            print("Implement train_mcsvm to see the accuracy.")
    else:
        print("The split helper lives in this Part's worked example — run it first.")
else:
    print("Finish Problem 6.1 first to power this demo.")

<details>
<summary>💡 Hint 1 — fit once, score thrice</summary>

One line fits the model: SVC(kernel="rbf", C=C, gamma=gamma).fit(X_train, y_train). model.score(X, y) gives each accuracy; the errors are 1 minus them. Keep the fitted model in the dict under 'model'.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
model = SVC(kernel="rbf", C=C, gamma=gamma).fit(X_train, y_train)
train_acc = float(model.score(X_train, y_train))  # same for val/test
return {"model": model, "train_acc": train_acc, "val_acc": val_acc,
        "test_acc": test_acc, "train_error": 1.0 - train_acc,
        "val_error": 1.0 - val_acc}
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def train_mcsvm(X_train, y_train, X_val, y_val, X_test, y_test, C=10.0, gamma="scale"):
    """Train a multi-class SVM (RBF) and report accuracies / errors."""
    model = SVC(kernel="rbf", C=C, gamma=gamma).fit(X_train, y_train)
    train_acc = float(model.score(X_train, y_train))
    val_acc = float(model.score(X_val, y_val))
    test_acc = float(model.score(X_test, y_test))
    return {"model": model, "train_acc": train_acc, "val_acc": val_acc,
            "test_acc": test_acc, "train_error": 1.0 - train_acc,
            "val_error": 1.0 - val_acc}
```
</details>

In [ ]:
# 🧪 Self-check for Problem 6.3
data63 = generate_four_moons(100, seed=3)
assert data63 is not None, "❌ generate_four_moons returned None — finish Problem 6.1 first."
X63c, y63c = data63
splits63 = train_validation_test_split(X63c, y63c, seed=3)
assert splits63 is not None, "❌ train_validation_test_split is missing — run this Part's worked example cell first."
(Xs63, ys63), (Xv63, yv63), (Xt63, yt63) = splits63
rep63 = train_mcsvm(Xs63, ys63, Xv63, yv63, Xt63, yt63)
assert rep63 is not None, "❌ train_mcsvm returned None — replace the 'pass'. See Hint 2 in Part 2."
for k in ("model", "train_acc", "val_acc", "test_acc", "train_error", "val_error"):
    assert k in rep63, f"❌ Dict is missing '{k}'."
for k in ("train_acc", "val_acc", "test_acc"):
    assert 0.0 <= rep63[k] <= 1.0, f"❌ {k} must be a fraction in [0, 1] — use model.score(X, y)."
assert abs(rep63["train_error"] - (1.0 - rep63["train_acc"])) < 1e-12, "❌ train_error must equal 1 - train_acc."
assert abs(rep63["val_error"] - (1.0 - rep63["val_acc"])) < 1e-12, "❌ val_error must equal 1 - val_acc."
assert hasattr(rep63["model"], "predict"), "❌ 'model' must be the fitted SVC (it has .predict)."
assert rep63["test_acc"] >= 0.9, f"❌ RBF SVM test accuracy should be >= 0.9, got {rep63['test_acc']:.3f} — check kernel='rbf' and that you fit on the TRAINING set only."
print("✅ Problem 6.3 passed — the SVM is trained and graded.")

## Part 3: A softmax neural network in pure numpy (≈12 min)

For $K$ classes the network outputs $K$ scores $Z_2$; the **softmax** turns them into probabilities $P_{ik} = e^{Z_{ik}} / \sum_j e^{Z_{ij}}$. The matching loss is **cross-entropy**: $L = -\tfrac{1}{n}\sum_i \log P_{i, y_i}$.

- **Standardize the inputs** ($x \mapsto (x - \mu)/\sigma$ using the *training-set* statistics): sigmoid layers train badly when features span tens of units.
- Backward pass: $dZ_2 = (P - Y)/n$ — cross-entropy + softmax simplify beautifully — then the same hidden-layer chain as Lab Activity 5.
- **Momentum** ($\mu = 0.95$) and a couple thousand epochs converge in seconds.

In [ ]:
# --- Worked example: softmax + cross-entropy by hand ---
def softmax_demo(Z):
    Z = Z - Z.max(axis=1, keepdims=True)      # shift for numerical stability
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)

P = softmax_demo(np.array([[2.0, 1.0, 0.1, 0.0],
                           [0.5, 0.5, 2.0, 0.1]]))
print("rows sum to 1:", bool(np.allclose(P.sum(axis=1), 1.0)))
print("probabilities:\n", np.round(P, 3))

Y_true = np.zeros((2, 4)); Y_true[0, 0] = 1.0; Y_true[1, 2] = 1.0   # labels [0, 2] one-hot
ce = -np.mean(np.sum(Y_true * np.log(P + 1e-12), axis=1))
print("cross-entropy of this batch:", round(float(ce), 4))

## 🎯 Problem 6.4 — The softmax MLNN class

**Given:** the architecture in the concept text.

**Required:** write `class MLNN(hidden=48, lr=0.1, epochs=3000, momentum=0.95, seed=0)` with `fit(X, y, X_val=None, y_val=None, num_classes=4)` / `predict` / `accuracy` and per-epoch cross-entropy `train_loss_` / `val_loss_`:

- standardize with the **training-set** mean/std (store them as `self.mu_`, `self.sd_` — `predict` must apply the same transform),
- hidden activation sigmoid with clipped arguments; softmax output with the max-shift trick,
- loss: cross-entropy against `one_hot(y)`; gradient $dZ_2 = (P - Y)/n$; hidden-layer chain and momentum as in Lab Activity 5,
- `predict`: argmax over the class scores.

**Expected output:** on the four-moon data the cross-entropy falls from ~1.6 below 0.4 and test accuracy reaches the low/mid 0.9s — competitive with (or better than) the SVM.

In [ ]:
class MLNN:
    """One-hidden-layer MLP with softmax output, pure NumPy."""

    def __init__(self, hidden=48, lr=0.1, epochs=3000, momentum=0.95, seed=0):
        self.hidden = hidden
        self.lr = lr
        self.epochs = epochs
        self.momentum = momentum
        self.seed = seed

    def fit(self, X, y, X_val=None, y_val=None, num_classes=4):
        """Train the network; record cross-entropy losses in train_loss_ / val_loss_."""
        # TODO: Your code here (forward pass, softmax, backpropagation, momentum update)
        pass

    def predict(self, X):
        """Return predicted class labels in {0, 1, 2, 3}."""
        # TODO: Your code here
        pass

    def accuracy(self, X, y):
        """Return the fraction of correct predictions."""
        # TODO: Your code here
        pass

# Demo call (uses your data + split once implemented)
demo_data64 = generate_four_moons(80, seed=3)
if demo_data64 is not None:
    qX, qy = demo_data64
    qsp = train_validation_test_split(qX, qy, seed=3)
    if qsp is not None:
        (qXt, qyt), (qXv, qyv), (_, _) = qsp
        demo_nn4 = MLNN(hidden=16, epochs=400).fit(qXt, qyt, qXv, qyv)
        if getattr(demo_nn4, "train_loss_", None) is not None:
            print("softmax loss first/last:", round(demo_nn4.train_loss_[0], 4), "/", round(demo_nn4.train_loss_[-1], 4))
        else:
            print("Implement MLNN.fit to see the loss curve.")
    else:
        print("Run Part 2's worked example cell first to power this demo.")
else:
    print("Finish Problem 6.1 first to power this demo.")

<details>
<summary>💡 Hint 1 — standardize, softmax, (P - Y)/n</summary>

Three extras compared to Lab 5's MLNN: (1) standardize X with the training mean/std and store them; (2) softmax output with the row-max shift; (3) the beautiful gradient dZ2 = (P - Y)/n. The hidden-layer chain and momentum are identical to Lab Activity 5.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
self.mu_ = X.mean(0); self.sd_ = X.std(0) + 1e-12; Xs = (X - mu) / sd
loop epochs:
    A1 = sigmoid(Xs @ W1 + b1)
    Z2 = A1 @ W2 + b2; Z2 -= Z2.max(axis=1, keepdims=True)
    P = exp(Z2); P /= P.sum(axis=1, keepdims=True)
    append cross-entropy -mean(sum(Y * log(P + 1e-12), axis=1))
    dZ2 = (P - Y) / n
    dW2 = A1.T @ dZ2; db2 = dZ2.sum(axis=0)
    dZ1 = (dZ2 @ W2.T) * A1 * (1 - A1); dW1 = Xs.T @ dZ1; db1 = dZ1.sum(axis=0)
    momentum update on all four parameters
predict: argmax(A1 @ W2 + b2, axis=1) on freshly standardized X
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
class MLNN:
    """One-hidden-layer MLP with softmax output, pure NumPy."""

    def __init__(self, hidden=48, lr=0.1, epochs=3000, momentum=0.95, seed=0):
        self.hidden = hidden
        self.lr = lr
        self.epochs = epochs
        self.momentum = momentum
        self.seed = seed

    def fit(self, X, y, X_val=None, y_val=None, num_classes=4):
        rng = np.random.default_rng(self.seed)
        n, d = X.shape
        self.mu_ = X.mean(axis=0); self.sd_ = X.std(axis=0) + 1e-12
        Xs = (X - self.mu_) / self.sd_
        Y = one_hot(y, num_classes)
        W1 = rng.normal(0, np.sqrt(1 / d), size=(d, self.hidden)); b1 = np.zeros(self.hidden)
        W2 = rng.normal(0, np.sqrt(1 / self.hidden), size=(self.hidden, num_classes)); b2 = np.zeros(num_classes)
        vW1 = np.zeros_like(W1); vb1 = np.zeros_like(b1)
        vW2 = np.zeros_like(W2); vb2 = np.zeros_like(b2)
        self.train_loss_, self.val_loss_ = [], []
        for _ in range(self.epochs):
            A1 = 1.0 / (1.0 + np.exp(-np.clip(Xs @ W1 + b1, -500, 500)))
            Z2 = A1 @ W2 + b2
            Z2 -= Z2.max(axis=1, keepdims=True)
            P = np.exp(Z2); P /= P.sum(axis=1, keepdims=True)
            self.train_loss_.append(float(-np.mean(np.sum(Y * np.log(P + 1e-12), axis=1))))
            if X_val is not None:
                Av = 1.0 / (1.0 + np.exp(-np.clip((X_val - self.mu_) / self.sd_ @ W1 + b1, -500, 500)))
                Zv = Av @ W2 + b2; Zv -= Zv.max(axis=1, keepdims=True)
                Pv = np.exp(Zv); Pv /= Pv.sum(axis=1, keepdims=True)
                self.val_loss_.append(float(-np.mean(np.log(Pv[np.arange(len(y_val)), y_val] + 1e-12))))
            dZ2 = (P - Y) / n
            dW2 = A1.T @ dZ2; db2 = dZ2.sum(axis=0)
            dZ1 = (dZ2 @ W2.T) * A1 * (1 - A1)
            dW1 = Xs.T @ dZ1; db1 = dZ1.sum(axis=0)
            vW1 = self.momentum * vW1 - self.lr * dW1; vb1 = self.momentum * vb1 - self.lr * db1
            vW2 = self.momentum * vW2 - self.lr * dW2; vb2 = self.momentum * vb2 - self.lr * db2
            W1 += vW1; b1 += vb1; W2 += vW2; b2 += vb2
        self.W1, self.b1, self.W2, self.b2 = W1, b1, W2, b2
        return self

    def predict(self, X):
        Xs = (X - self.mu_) / self.sd_
        A1 = 1.0 / (1.0 + np.exp(-np.clip(Xs @ self.W1 + self.b1, -500, 500)))
        return np.argmax(A1 @ self.W2 + self.b2, axis=1)

    def accuracy(self, X, y):
        return float(np.mean(self.predict(X) == y))
```
</details>

In [ ]:
# 🧪 Self-check for Problem 6.4
data64 = generate_four_moons(100, seed=3)
assert data64 is not None, "❌ generate_four_moons returned None — finish Problem 6.1 first."
X64, y64 = data64
splits64 = train_validation_test_split(X64, y64, seed=3)
assert splits64 is not None, "❌ train_validation_test_split is missing — run Part 2's worked example cell first."
(Xs64, ys64), (Xv64, yv64), (Xt64, yt64) = splits64
net64 = MLNN(hidden=48, lr=0.1, epochs=1500, momentum=0.95, seed=0)
out64 = net64.fit(Xs64, ys64, Xv64, yv64)
assert out64 is not None, "❌ fit returned None — replace the 'pass'. See Hint 2 in Part 3 (and return self)."
tl64 = getattr(net64, "train_loss_", None)
assert tl64 is not None and len(tl64) == 1500, "❌ Record one cross-entropy training loss per epoch in self.train_loss_."
assert all(np.isfinite(tl64)), "❌ Losses must be finite — clip sigmoid arguments and shift softmax logits by their row max."
assert tl64[-1] < tl64[0], "❌ The cross-entropy must decrease — check the sign of the gradient step."
assert tl64[-1] < 0.6, f"❌ Final cross-entropy should drop below 0.6, got {tl64[-1]:.3f} — check dZ2 = (P - Y)/n and the momentum update."
assert hasattr(net64, "mu_") and hasattr(net64, "sd_"), "❌ Store the training-set mean/std as self.mu_ / self.sd_ — predict depends on them."
pr64 = net64.predict(Xt64)
assert set(np.unique(pr64)).issubset({0, 1, 2, 3}), "❌ predict must return labels in {0, 1, 2, 3}."
acc64 = net64.accuracy(Xt64, yt64)
assert acc64 >= 0.85, f"❌ Test accuracy should reach >= 0.85, got {acc64:.3f} — check standardization (fit on the TRAINING set only) and the softmax gradient."
assert net64.accuracy(Xs64, ys64) >= 0.85, "❌ The network should fit the training set well above 0.85 — training is not converging."
print("✅ Problem 6.4 passed — the softmax network learns the four moons.")

## Part 4: Comparing SVM vs MLNN & mini-challenge (≈12 min)

Both models are trained — how do they differ?

- **Error/loss curves**: the MLNN has a per-epoch curve (a convergence story); the SVM reports a single final error. Plot both on one axis to compare where they land.
- **Per-class behavior**: overall accuracy hides which crescent is misclassified. Compute the accuracy within each class — per-class numbers explain *what* a model gets wrong.
- **Decision regions**: predict over a fine grid and `contourf` — the SVM draws smooth blobs around each crescent, the MLNN similar shapes with slightly different boundaries.

**Mini-challenge:** replicate the assignment's full comparison workflow end-to-end.

In [ ]:
# --- Worked example: what does a classifier get wrong? per-class accuracy ---
for cls in (0, 1):
    mask = ydemo == cls
    print(f"blob demo, class {cls}: {svm_demo.score(Xdemo[mask], ydemo[mask]):.3f}")
print("overall:", round(float(svm_demo.score(Xdemo, ydemo)), 3),
      " <- per-class numbers can be lower where the classes nearly touch")

## 🎯 Problem 6.5 — compare_classifiers

**Given:** an SVM report dict (as returned by `train_mcsvm`) and the MLNN's test accuracy.

**Required:** write `compare_classifiers(svm_report, mlnn_test_acc)` returning a dict with keys:

- `'svm_test_acc'`, `'mlnn_test_acc'` — the two test accuracies (floats),
- `'better'` — `"MLNN"` if the MLNN is strictly better, `"MCSVM"` if the SVM is strictly better, else `"tie"`,
- `'margin'` — the absolute accuracy difference (float).

**Expected output:** `margin` quantifies how close the race is; `better` makes the verdict explicit. The assignment's Task 4 wants exactly this dict.

In [ ]:
def compare_classifiers(svm_report, mlnn_test_acc):
    """Compare the MCSVM and the MLNN on test accuracy.

    Args:
        svm_report: dict as returned by train_mcsvm
        mlnn_test_acc: float, MLNN test accuracy

    Returns:
        dict with keys 'svm_test_acc', 'mlnn_test_acc', 'better', 'margin'
    """
    # TODO: Your code here
    pass

# Demo call
fake_report = {"test_acc": 0.94, "train_acc": 0.97, "val_acc": 0.93,
               "train_error": 0.03, "val_error": 0.07, "model": None}
demo_verdict = compare_classifiers(fake_report, 0.95)
if demo_verdict is not None:
    print("verdict:", demo_verdict)
else:
    print("Implement compare_classifiers to power this demo.")

<details>
<summary>💡 Hint 1 — read, compare, measure</summary>

Pull svm_report['test_acc'], compare the two floats with if/elif/else for the three verdicts, and margin is abs(mlnn - svm). No models involved — just dict arithmetic.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
svm_acc = float(svm_report["test_acc"]); mlnn_acc = float(mlnn_test_acc)
better = "MLNN" if mlnn_acc > svm_acc else ("MCSVM" if svm_acc > mlnn_acc else "tie")
return {"svm_test_acc": svm_acc, "mlnn_test_acc": mlnn_acc,
        "better": better, "margin": abs(mlnn_acc - svm_acc)}
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def compare_classifiers(svm_report, mlnn_test_acc):
    """Compare the MCSVM and the MLNN on test accuracy."""
    svm_acc = float(svm_report["test_acc"])
    mlnn_acc = float(mlnn_test_acc)
    if mlnn_acc > svm_acc:
        better = "MLNN"
    elif svm_acc > mlnn_acc:
        better = "MCSVM"
    else:
        better = "tie"
    return {"svm_test_acc": svm_acc, "mlnn_test_acc": mlnn_acc,
            "better": better, "margin": abs(mlnn_acc - svm_acc)}
```
</details>

In [ ]:
# 🧪 Self-check for Problem 6.5
rep65 = {"model": None, "train_acc": 0.97, "val_acc": 0.93, "test_acc": 0.94,
         "train_error": 0.03, "val_error": 0.07}
out65 = compare_classifiers(rep65, 0.95)
assert out65 is not None, "❌ compare_classifiers returned None — replace the 'pass'. See Hint 2 in Part 4."
for k in ("svm_test_acc", "mlnn_test_acc", "better", "margin"):
    assert k in out65, f"❌ Dict is missing '{k}'."
assert abs(out65["svm_test_acc"] - 0.94) < 1e-12, "❌ svm_test_acc must come from svm_report['test_acc']."
assert abs(out65["mlnn_test_acc"] - 0.95) < 1e-12, "❌ mlnn_test_acc must be the second argument."
assert out65["better"] == "MLNN", "❌ With 0.95 > 0.94 the verdict must be 'MLNN'."
assert abs(out65["margin"] - 0.01) < 1e-12, "❌ margin must be the absolute difference between the two accuracies."
out65b = compare_classifiers(rep65, 0.90)
assert out65b["better"] == "MCSVM", "❌ With 0.90 < 0.94 the verdict must be 'MCSVM'."
out65c = compare_classifiers(rep65, 0.94)
assert out65c["better"] == "tie" and abs(out65c["margin"]) < 1e-12, "❌ Equal accuracies must give 'tie' and a zero margin."
print("✅ Problem 6.5 passed — verdicts are consistent.")

## 🎯 Problem 6.6 — Mini-challenge: the full assignment workflow

**Given:** everything from this lab.

**Required:** write `run_full_comparison(n_per_class=100, seed=5)` that:

1. generates the four-moon data and splits it 60/20/20 with the same `seed`,
2. trains the RBF SVM (`C=10.0`) and the softmax MLNN (`hidden=48`, `epochs=1500`) on the same training set,
3. builds a 1×2 decision-region figure: predict each model over a fine grid covering the data, `contourf` the predicted classes (4 levels), scatter the test points,
4. returns a dict with keys `'svm_test_acc'`, `'mlnn_test_acc'`, `'better'`, `'figure'` — `better` uses the same rule as Problem 6.5.

**Expected output:** both models land in the low/mid 0.9s on the test set; the two panels show how each carves the plane into four crescent-shaped regions. This is **Lab Assignment 06** Tasks 1–4 in a single function.

In [ ]:
def run_full_comparison(n_per_class=100, seed=5):
    """Replicate the assignment workflow: data -> split -> SVM vs MLNN -> figure.

    Args:
        n_per_class: points per category
        seed: reproducibility seed for generation and splitting

    Returns:
        dict with keys 'svm_test_acc', 'mlnn_test_acc', 'better', 'figure'
    """
    # TODO: Your code here
    pass

# Demo call
demo_full = run_full_comparison()
if demo_full is not None:
    print("SVM:", round(demo_full["svm_test_acc"], 3),
          "| MLNN:", round(demo_full["mlnn_test_acc"], 3),
          "| better:", demo_full["better"])
    plt.show()
else:
    print("Implement run_full_comparison to power this demo.")

<details>
<summary>💡 Hint 1 — assemble your own tools</summary>

Every step exists already: generate_four_moons, train_validation_test_split, train_mcsvm, MLNN, compare logic. The only new piece is the 1x2 decision-region figure: meshgrid over the data extent, predict on the raveled grid, contourf with levels=4, scatter the test points.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
X, y = generate_four_moons(n_per_class, seed=seed)
split once with the same seed
svm_rep = train_mcsvm(X_train, ..., C=10.0)
net = MLNN(hidden=48, epochs=1500).fit(X_train, y_train, X_val, y_val)
GX, GY = meshgrid over [min-1, max+1]; grid_pts = column_stack([GX.ravel(), GY.ravel()])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0]: contourf(GX, GY, svm_rep["model"].predict(grid_pts).reshape(GX.shape), levels=4)
axes[1]: same with net.predict
return dict with both test accuracies, the verdict and the figure
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def run_full_comparison(n_per_class=100, seed=5):
    """Replicate the assignment workflow: data -> split -> SVM vs MLNN -> figure."""
    X, y = generate_four_moons(n_per_class, seed=seed)
    (X_train, y_train), (X_val, y_val), (X_test, y_test) = train_validation_test_split(X, y, seed=seed)

    svm_rep = train_mcsvm(X_train, y_train, X_val, y_val, X_test, y_test, C=10.0)
    net = MLNN(hidden=48, lr=0.1, epochs=1500, momentum=0.95, seed=0)
    net.fit(X_train, y_train, X_val, y_val)
    mlnn_acc = float(net.accuracy(X_test, y_test))

    gx = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 250)
    gy = np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 250)
    GX, GY = np.meshgrid(gx, gy)
    grid_pts = np.column_stack([GX.ravel(), GY.ravel()])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, preds, title in [(axes[0], svm_rep["model"].predict(grid_pts), "MCSVM (RBF)"),
                             (axes[1], net.predict(grid_pts), "MLNN (softmax)")]:
        ax.contourf(GX, GY, preds.reshape(GX.shape), levels=4, alpha=0.35, cmap="tab10")
        ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, s=8, cmap="tab10")
        ax.set_title(title)
    fig.tight_layout()

    svm_acc = float(svm_rep["test_acc"])
    if mlnn_acc > svm_acc:
        better = "MLNN"
    elif svm_acc > mlnn_acc:
        better = "MCSVM"
    else:
        better = "tie"
    return {"svm_test_acc": svm_acc, "mlnn_test_acc": mlnn_acc,
            "better": better, "figure": fig}
```
</details>

In [ ]:
# 🧪 Self-check for Problem 6.6
full = run_full_comparison()
assert full is not None, "❌ run_full_comparison returned None — replace the 'pass'. See Hint 2 in Part 4."
for k in ("svm_test_acc", "mlnn_test_acc", "better", "figure"):
    assert k in full, f"❌ Dict is missing '{k}'."
assert 0.8 <= full["svm_test_acc"] <= 1.0, f"❌ SVM test accuracy looks off ({full['svm_test_acc']:.3f}) — fit on the training split only, with kernel='rbf'."
assert 0.8 <= full["mlnn_test_acc"] <= 1.0, f"❌ MLNN test accuracy looks off ({full['mlnn_test_acc']:.3f}) — standardize with the training statistics and use momentum."
expected66 = "MLNN" if full["mlnn_test_acc"] > full["svm_test_acc"] else ("MCSVM" if full["svm_test_acc"] > full["mlnn_test_acc"] else "tie")
assert full["better"] == expected66, "❌ 'better' must follow the same rule as Problem 6.5."
import matplotlib.figure as _mpl_fig6
import matplotlib.collections as _mpl_col6
from matplotlib.contour import QuadContourSet as _QCS6

def _has_contourf6(ax):
    # matplotlib >= 3.8 registers contourf as a QuadContourSet in ax.collections;
    # older versions register a QuadMesh. Accept either.
    return any(isinstance(c, (_mpl_col6.QuadMesh, _QCS6)) for c in ax.collections)

fig6 = full["figure"]
assert isinstance(fig6, _mpl_fig6.Figure), "❌ 'figure' must be a matplotlib Figure."
panels6 = fig6.get_axes()[:2]
assert len(panels6) == 2, "❌ The figure needs 2 side-by-side decision-region panels."
for i, ax in enumerate(panels6):
    assert _has_contourf6(ax), f"❌ Panel {i} needs a contourf of the predicted classes over the grid."
print("✅ Problem 6.6 passed — SVM vs MLNN, end to end. Lab Assignment 06 awaits!")

## 🎉 You've completed Lab Activity 6

You have mastered:
- Generating four interleaved crescents (two moon pairs side by side) and one-hot encoding labels
- Training a multi-class SVM (`SVC`, RBF kernel) and grading it on held-out splits
- Building a softmax neural network in pure numpy — standardization, cross-entropy, momentum
- Comparing models properly: overall accuracy, per-class accuracy, and decision regions

**You are now ready for Lab Assignment 06 on the course portal — the assignment asks for the same techniques without hints.**

💡 **Tip:** restart the kernel and run every cell top-to-bottom once more — each 🧪 self-check should print ✅. Then try `C=1.0` and `C=100.0` on the SVM: the decision regions will show the regularization dial in action.